# Practice: evaluating an agent

You evaluate a small travel-money agent with two tools: a currency converter with a fixed rate table and a percentage calculator. You write ten tasks with their expected outcomes, grade the agent's answers with code, check from the transcript that it used the right tools, run every task three times to measure how reliable it is, and grade free-text answers with Claude as a judge, which you then check against your own labels.

After this workbook you can build a small evaluation suite for an agent, choose between a code grader and a model grader, compute pass@k and pass^k, and say how far a judge can be trusted.

> **Note.** The saved outputs come from a run on 3 November 2026 with Claude Haiku 5.5. Cells that call Claude carry the tag `needs-api-key`: they need your key in `.env` (see *Setup*), and their answers change from run to run.

## Overview

| Section | What it covers |
|---|---|
| Setup: key and model | The key from `.env`, Claude Haiku 5.5 |
| The agent under test | Two tools with a fixed rate table, `create_agent` |
| A golden dataset of tasks | The words task, trial, grader, transcript and outcome; ten tasks as a list of dictionaries |
| An outcome grader in code | Checking the number or the words in the final answer, testing the grader |
| A tool-use grader that reads the transcript | Which tools the agent called, a task where no tool is right |
| Three trials: pass@k and pass^k | Running every task three times, at least once against every time |
| An LLM judge for free-text answers | A binary judge with a critique first, for answers code cannot grade |
| Checking the judge against hand labels | Agreement, true positive rate and true negative rate |
| Check your understanding | Four questions to vote on |
| Exercises | Practice in pairs |
| Debrief | What to compare after the exercises |
| Further reading | The sources of the vocabulary and the metrics |

## Setup: key and model

All packages are in the course environment (`uv sync`). The key lives in `.env` in the `workbooks` folder of this session, with the line `ANTHROPIC_API_KEY=...`; `.env.example` shows the format and git ignores `.env`. The cell finds the folder whether Jupyter started in it or at the repository root; `find_dotenv(usecwd=True)` is the fallback that searches upwards from the current folder.

In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

candidates = [Path.cwd(), Path.cwd() / "sessions" / "15-rag-and-agents" / "workbooks"]
WORKBOOKS = next(folder for folder in candidates if folder.parts[-2:] == ("15-rag-and-agents", "workbooks"))
env_file = WORKBOOKS / ".env" if (WORKBOOKS / ".env").is_file() else find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Key loaded." if HAS_KEY else "No ANTHROPIC_API_KEY found: copy .env.example to .env in the workbooks folder and add your key. Only the cells that call Claude need it.")

Key loaded.


The agent and the judge use **Claude Haiku 5.5** (`claude-haiku-5-5`) through `ChatAnthropic`. Always set `max_tokens`: without it, `langchain-anthropic` asks for the full limit of 128,000 tokens. An evaluation calls the model many times (here about 70 calls), so the cheapest model that does the job is the right start. This cell needs the key.

In [2]:
from langchain_anthropic import ChatAnthropic

if not HAS_KEY:
    raise RuntimeError("Add ANTHROPIC_API_KEY to the .env file in the workbooks folder first (see Setup).")
model = ChatAnthropic(model="claude-haiku-5-5", max_tokens=1024, timeout=60, max_retries=2)

## The agent under test

The agent helps with travel money. It has two tools:

- `convert_currency` converts with a **fixed rate table**. The rates are made up for this workbook and never change, so the correct answer to a task stays the same. With live rates, the expected answers would change every day, and a failed task could mean a wrong agent or just a new rate.
- `percent_of` computes a percentage, for tips, discounts and price rises.

`@tool(parse_docstring=True)` from `langchain.tools` turns a function into a tool, and the `Args:` section of the docstring describes each argument for the model. Each tool returns a short JSON text, or a sentence that starts with `Error:`.

In [3]:
import json

from langchain.tools import tool

RATES_PER_EUR = {"EUR": 1.0, "USD": 1.16, "GBP": 0.87, "CZK": 24.30, "PLN": 4.25, "CHF": 0.93, "JPY": 172.0}


@tool(parse_docstring=True)
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between two currencies with the fixed rate table of this workbook.

    Args:
        amount: The amount of money in from_currency.
        from_currency: ISO 4217 code, for example "EUR".
        to_currency: ISO 4217 code, for example "CZK".
    """
    source, target = from_currency.upper(), to_currency.upper()
    missing = [code for code in (source, target) if code not in RATES_PER_EUR]
    if missing:
        return f"Error: no rate for {', '.join(missing)}. Rates exist only for {', '.join(RATES_PER_EUR)}."
    result = amount / RATES_PER_EUR[source] * RATES_PER_EUR[target]
    return json.dumps({"amount": round(result, 2), "currency": target})


@tool(parse_docstring=True)
def percent_of(value: float, percent: float) -> str:
    """Compute a percentage of a value, for tips, discounts and price rises.

    Args:
        value: The base value, for example a price.
        percent: The percentage, for example 15 for 15 %.
    """
    return json.dumps({"result": round(value * percent / 100, 2)})

**Predict.** How many Polish zloty (PLN) are 1,000 Czech koruna (CZK) with this table? And what does the tool answer for Bitcoin (`BTC`)?

In [4]:
print(convert_currency.invoke({"amount": 1000, "from_currency": "CZK", "to_currency": "PLN"}))
print(convert_currency.invoke({"amount": 100, "from_currency": "EUR", "to_currency": "BTC"}))
print(percent_of.invoke({"value": 80, "percent": 15}))

{"amount": 174.9, "currency": "PLN"}
Error: no rate for BTC. Rates exist only for EUR, USD, GBP, CZK, PLN, CHF, JPY.
{"result": 12.0}


1,000 CZK is 1000 / 24.30 = 41.15 EUR, which is 41.15 × 4.25 = 174.90 PLN: the tool goes through the euro. For Bitcoin there is no rate, and the tool says so instead of guessing.

The agent itself is one `create_agent` call. The system prompt asks it to use the tools for every calculation and to say so when a currency is missing. This cell needs the key.

In [5]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "You help travellers with money questions. Use the tools for every conversion and percentage; "
    "never compute in your head. If a tool has no rate for a currency, say so and do not estimate. "
    "Answer in one or two sentences and give amounts with two decimals."
)
agent = create_agent(model=model, tools=[convert_currency, percent_of], system_prompt=SYSTEM_PROMPT)

## A golden dataset of tasks

### Task, trial, grader, transcript, outcome

Anthropic's guide *Demystifying evals for AI agents* (January 2026) uses these words, and the course slides use them too:

| Word | Meaning | Here |
|---|---|---|
| **Task** | One test case with an input and success criteria | "How much is 50 EUR in Czech koruna?", expected 1,215.00 |
| **Trial** | One attempt of the agent at a task. Agents are not deterministic, so a task gets several trials | three trials per task |
| **Grader** | Code, a model or a person that scores one aspect of a trial | the outcome grader, the tool-use grader, the judge |
| **Transcript** | Everything that happened in a trial: messages, tool calls, results. Also called trace or trajectory | `result["messages"]` |
| **Outcome** | The final result or state at the end of a trial | the number in the answer |

The guide's example shows why outcome and transcript differ: a flight-booking agent may say "Your flight has been booked", but the outcome is whether a reservation exists in the database.

A **golden dataset** is the fixed set of tasks with their expected outcomes. "Golden" means the expected answers were checked by a person. It does not have to be large at the start: the guide calls 20 to 50 tasks drawn from real failures "a great start". Ours has ten, as a list of dictionaries:

- `input`: the user's message.
- `grader`: `"number"` (the answer must contain the expected number), `"text"` (it must contain the expected words) or `"judge"` (a model grades it).
- `expected`: the number or the words; `None` for judge tasks.
- `required_tools`: the tools the agent must call. An empty list means the agent must call **no** tool, and `None` means we do not grade tool use for that task.

Two tasks test what should *not* happen: `t09` needs no tool, and `t10` asks for a currency without a rate. Such rows matter: in the guide's words, "one-sided evals create one-sided optimization".

In [6]:
TASKS = [
    {"id": "t01", "input": "How much is 50 EUR in Czech koruna?", "grader": "number", "expected": 1215.00, "required_tools": ["convert_currency"]},
    {"id": "t02", "input": "Convert 200 US dollars to euros.", "grader": "number", "expected": 172.41, "required_tools": ["convert_currency"]},
    {"id": "t03", "input": "Our dinner in Berlin cost 80 EUR. How much is a 15 % tip?", "grader": "number", "expected": 12.00, "required_tools": ["percent_of"]},
    {"id": "t04", "input": "A jacket costs 120 EUR with 25 % off. How many euros do I save?", "grader": "number", "expected": 30.00, "required_tools": ["percent_of"]},
    {"id": "t05", "input": "I have 1000 CZK left after a trip. How many Polish zloty is that?", "grader": "number", "expected": 174.90, "required_tools": ["convert_currency"]},
    {"id": "t06", "input": "My rent of 450 GBP goes up by 3 %. By how many pounds?", "grader": "number", "expected": 13.50, "required_tools": ["percent_of"]},
    {"id": "t07", "input": "What is 1000 Japanese yen in euros?", "grader": "number", "expected": 5.81, "required_tools": ["convert_currency"]},
    {"id": "t08", "input": "How much is 30 % of 250 GBP, in euros?", "grader": "number", "expected": 86.21, "required_tools": ["convert_currency", "percent_of"]},
    {"id": "t09", "input": "What does the currency code CHF stand for?", "grader": "text", "expected": "swiss franc", "required_tools": []},
    {"id": "t10", "input": "Convert 100 EUR to Bitcoin.", "grader": "judge", "expected": None, "required_tools": None},
]

`pandas` shows the list as a table. `pd.DataFrame(list_of_dicts)` makes one row per dictionary and one column per key.

**Predict.** Which grader will each of the ten tasks use, and how many tasks require exactly one tool?

In [7]:
import pandas as pd

pd.DataFrame(TASKS)[["id", "grader", "expected", "required_tools"]]

,id,grader,expected,required_tools
0,t01,number,1215.0,[convert_currency]
1,t02,number,172.41,[convert_currency]
2,t03,number,12.0,[percent_of]
3,t04,number,30.0,[percent_of]
4,t05,number,174.9,[convert_currency]
5,t06,number,13.5,[percent_of]
6,t07,number,5.81,[convert_currency]
7,t08,number,86.21,"[convert_currency, percent_of]"
8,t09,text,swiss franc,[]
9,t10,judge,None,None


Seven tasks require exactly one tool; `t08` requires both, `t09` none, and `t10` is not graded on tools. A golden dataset is software and can have bugs, so check the expected values before you trust them. Here the tools themselves can compute them, independent of any model:

In [8]:
print(convert_currency.invoke({"amount": 50, "from_currency": "EUR", "to_currency": "CZK"}))
print(convert_currency.invoke({"amount": 200, "from_currency": "USD", "to_currency": "EUR"}))
print(convert_currency.invoke({"amount": 1000, "from_currency": "JPY", "to_currency": "EUR"}))
print(percent_of.invoke({"value": 450, "percent": 3}))
print(convert_currency.invoke({"amount": 75, "from_currency": "GBP", "to_currency": "EUR"}))

{"amount": 1215.0, "currency": "CZK"}
{"amount": 172.41, "currency": "EUR"}
{"amount": 5.81, "currency": "EUR"}
{"result": 13.5}
{"amount": 86.21, "currency": "EUR"}


## An outcome grader in code

The **outcome grader** looks only at the final answer. For number tasks it finds every number in the answer and passes the trial if one of them is within 0.02 of the expected value; the tolerance allows for rounding in the last digit (`t08` can be computed in two orders). For text tasks it checks that the expected words appear, ignoring case. Judge tasks return `None`: code cannot grade them.

`re.findall(pattern, text)` returns every match of a regular expression. The pattern `-?\d[\d,]*(?:\.\d+)?` reads: an optional minus, a digit, more digits or commas, and optionally a point with digits. `.replace(",", "")` removes the thousands separators before `float`.

In [9]:
import re

NUMBER = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def numbers_in(text: str) -> list[float]:
    """Return every number in the text, with thousands separators removed."""
    return [float(match.replace(",", "")) for match in NUMBER.findall(text)]


def grade_outcome(task: dict, answer: str) -> bool | None:
    """Pass if the answer contains the expected number (within 0.02) or the expected words."""
    if task["grader"] == "number":
        return any(abs(number - task["expected"]) <= 0.02 for number in numbers_in(answer))
    if task["grader"] == "text":
        return task["expected"] in answer.lower()
    return None

### Testing the grader

A grader is code, and code needs tests. The strings below are written by hand to test the grader; they are not model answers.

**Predict.** Which of these four answers to `t01` (expected 1,215.00 CZK) pass?

In [10]:
t01 = TASKS[0]
test_answers = [
    "50 EUR is 1,215.00 CZK.",
    "That is 1215 Czech koruna.",
    "50 EUR sind 1.215,00 CZK.",
    "At a rate of 24.30, 50 EUR gives 1,250.00 CZK.",
]
for answer in test_answers:
    print(grade_outcome(t01, answer), "|", answer, "|", numbers_in(answer))

True | 50 EUR is 1,215.00 CZK. | [50.0, 1215.0]
True | That is 1215 Czech koruna. | [1215.0]
False | 50 EUR sind 1.215,00 CZK. | [50.0, 1.215, 0.0]
False | At a rate of 24.30, 50 EUR gives 1,250.00 CZK. | [24.3, 50.0, 1250.0]


The first two pass. The third is correct, but in German number format: the point separates thousands and the comma separates decimals, so the grader reads `1.215` and `0` and fails it. The fourth fails rightly. This is the most common weakness of code graders: they are fast, cheap and repeatable, but brittle to valid variations of the form. The guide tells the story of a benchmark that rejected "96.12" when it expected "96.124991...". Two fixes: ask for a format in the system prompt (we did: two decimals), or make the grader accept both formats. Also note the opposite risk: an answer that lists many numbers might contain the right one by chance.

## A tool-use grader that reads the transcript

The **tool-use grader** reads the transcript instead of the answer. In LangChain, every reply of the model is an `AIMessage`, and `message.tool_calls` lists the tools it asked for, each as a dictionary with `name`, `args` and `id`. The grader passes if every required tool was called. For a task with an empty list it passes only if **no** tool was called.

`set(a) <= set(b)` is true if every element of `a` is also in `b`, so extra tool calls do not fail a task. This is deliberate: grading the exact path is often too strict, because agents find valid ways the task author did not expect. Grade the outcome first and fix the path only where the order is a rule, for example "check the identity before a refund".

In [11]:
from langchain.messages import AIMessage


def tools_called(messages: list) -> list[str]:
    """Return the names of all tools the model asked for, in order."""
    return [call["name"] for message in messages if isinstance(message, AIMessage) for call in message.tool_calls]


def grade_tool_use(task: dict, messages: list) -> bool | None:
    """Pass if every required tool was called; with an empty list, pass only if no tool was called."""
    if task["required_tools"] is None:
        return None
    called = set(tools_called(messages))
    if not task["required_tools"]:
        return not called
    return set(task["required_tools"]) <= called

**Predict.** The transcript below is written by hand to test the grader: the model answered `t03` (the 15 % tip) correctly, but computed it in its head. What do the two graders say?

In [12]:
from langchain.messages import HumanMessage

t03 = TASKS[2]
hand_written_transcript = [HumanMessage(t03["input"]), AIMessage("A 15 % tip on 80 EUR is 12.00 EUR.")]
print("outcome:", grade_outcome(t03, hand_written_transcript[-1].text))
print("tool use:", grade_tool_use(t03, hand_written_transcript))

outcome: True
tool use: False


The outcome passes, the tool use fails. Both are true: the answer is right this time, but the agent broke the rule "use the tools for every calculation", and mental arithmetic fails on harder numbers. This is why we grade the transcript as well as the outcome.

## Three trials: pass@k and pass^k

The same task can pass in one trial and fail in the next. Two numbers describe this, for a task with per-trial success rate $p$ and $k$ independent trials:

$$\text{pass@}k = 1 - (1 - p)^k \qquad \text{pass}^k = p^k$$

- **pass@k** is the probability that **at least one** of $k$ trials succeeds. It fits a tool where one good answer is enough, for example generating three code suggestions and keeping the one that works.
- **pass^k** ("pass hat k") is the probability that **all** $k$ trials succeed. It fits an agent that customers use again and again, where every run must be right. It was introduced with the τ-bench benchmark (Yao et al., 2024).

**Predict.** An agent passes a task in 75 % of trials. What are pass@3 and pass^3?

In [13]:
for p in (0.75, 0.90, 0.99):
    print(f"p = {p:.2f}   pass@3 = {1 - (1 - p) ** 3:.3f}   pass^3 = {p ** 3:.3f}   pass^10 = {p ** 10:.3f}")

p = 0.75   pass@3 = 0.984   pass^3 = 0.422   pass^10 = 0.056
p = 0.90   pass@3 = 0.999   pass^3 = 0.729   pass^10 = 0.349
p = 0.99   pass@3 = 1.000   pass^3 = 0.970   pass^10 = 0.904


With $p = 0.75$, pass@3 is 98.4 % but pass^3 only 42.2 %: the same agent looks almost perfect or unreliable, depending on the question. Ten trials push pass^10 down to 5.6 %. Even at $p = 0.99$, one user in ten who asks ten times sees a failure.

### Running the trials

`run_trial` runs the agent once on a task and keeps the transcript, the answer and the tool calls. `message.text` is the text of a reply as a string. The next cell runs every task three times: 30 trials, about 60 to 70 model calls. It needs the key.

In [14]:
TRIALS_PER_TASK = 3


def run_trial(task: dict) -> dict:
    """Run the agent once on a task and return its transcript, answer and tool calls."""
    result = agent.invoke({"messages": [{"role": "user", "content": task["input"]}]})
    messages = result["messages"]
    return {"task": task["id"], "transcript": messages, "answer": messages[-1].text, "tools": tools_called(messages)}


trials = [run_trial(task) for task in TASKS for _ in range(TRIALS_PER_TASK)]
print(len(trials), "trials")

30 trials


A trial passes if every grader that applies to it passes. `t10` has no code grader, so it is left out here and graded by the judge in the next section. The table shows one row per trial.

In [15]:
TASK_BY_ID = {task["id"]: task for task in TASKS}
rows = []
for trial in trials:
    task = TASK_BY_ID[trial["task"]]
    grades = [grade_outcome(task, trial["answer"]), grade_tool_use(task, trial["transcript"])]
    applicable = [grade for grade in grades if grade is not None]
    rows.append({"task": trial["task"], "outcome": grades[0], "tool_use": grades[1], "passed": all(applicable) if applicable else None, "tools": trial["tools"], "answer": trial["answer"][:80]})
results = pd.DataFrame(rows)
results

,task,outcome,tool_use,passed,tools,answer
0,t01,True,True,True,[convert_currency],50 EUR is 1215.00 CZK.
1,t01,True,True,True,[convert_currency],50 EUR is 1215.00 CZK.
2,t01,True,True,True,[convert_currency],50 EUR is 1215.00 CZK.
3,t02,True,True,True,[convert_currency],200 US dollars is 172.41 euros.
4,t02,True,True,True,[convert_currency],200 US dollars is 172.41 euros.
5,t02,True,True,True,[convert_currency],200 US dollars is 172.41 euros.
6,t03,True,True,True,[percent_of],A 15% tip on your 80.00 EUR dinner comes to 12...
7,t03,True,True,True,[percent_of],A 15% tip on your 80.00 EUR dinner is 12.00 EUR.
8,t03,True,True,True,[percent_of],A 15% tip on your 80.00 EUR dinner comes to 12...
9,t04,True,True,True,[percent_of],"You save 30.00 EUR on the jacket, which is 25 ..."


Now per task: the share of passed trials $\hat{p}$, **pass@3** measured as "at least one of the three passed", and **pass^3** measured as "all three passed". `groupby("task")["passed"]` groups the trials by task; `.mean()`, `.any()` and `.all()` summarise each group.

In [16]:
graded = results.dropna(subset=["passed"]).astype({"passed": bool})
per_task = graded.groupby("task")["passed"].agg(p_hat="mean", pass_at_3="any", pass_hat_3="all")
print(per_task)
print()
print(f"pass rate over all trials: {graded['passed'].mean():.2f}")
print(f"pass@3 over tasks:  {per_task['pass_at_3'].mean():.2f}")
print(f"pass^3 over tasks:  {per_task['pass_hat_3'].mean():.2f}")

      p_hat  pass_at_3  pass_hat_3
task                              
t01     1.0       True        True
t02     1.0       True        True
t03     1.0       True        True
t04     1.0       True        True
t05     1.0       True        True
t06     1.0       True        True
t07     1.0       True        True
t08     1.0       True        True
t09     1.0       True        True

pass rate over all trials: 1.00
pass@3 over tasks:  1.00
pass^3 over tasks:  1.00


Read the gap between pass@3 and pass^3: each task that passes at least once but not every time is a reliability problem, and its transcripts are where to look first. With nine tasks and three trials, these numbers are rough. Spend a larger budget first on more tasks, then on more trials per task, because three trials of the same task are not three independent pieces of evidence.

## An LLM judge for free-text answers

`t10` asks for Bitcoin, which has no rate. A good answer says clearly that no rate exists and gives no amount. There are many correct ways to say that, so a string match would be brittle. Here a **model grader**, or **LLM judge**, helps: another model call that reads the answer and decides. Use code graders where you can and a judge where you must.

The judge prompt follows common advice for judges:

- **Binary**: PASS or FAIL, no scale from 1 to 5. Neighbouring points on a scale are hard to tell apart, also for people.
- **One criterion** per judge.
- **Critique first, verdict last**: the model writes its reasons before it decides.
- **A way out**: UNKNOWN when the answer does not give enough information.
- **Only what it needs**: the question and the answer, not the whole transcript.

The verdict is read from the last line with a regular expression; anything else counts as UNKNOWN.

In [17]:
JUDGE_PROMPT = """You grade one answer of a currency assistant. The assistant has exchange rates only for EUR, USD, GBP, CZK, PLN, CHF and JPY.

Criterion: if the user asks about a currency without a rate, a good answer says clearly that there is no rate for it and gives no converted amount, not even an estimate.

Question: {question}
Answer: {answer}

First write a critique of two or three sentences about this criterion only.
Then write the last line exactly as VERDICT: PASS or VERDICT: FAIL. If the answer does not let you decide, write VERDICT: UNKNOWN."""

VERDICT = re.compile(r"VERDICT:\s*(PASS|FAIL|UNKNOWN)")


def parse_verdict(judge_reply: str) -> str:
    """Return PASS, FAIL or UNKNOWN from the judge's reply; anything unreadable counts as UNKNOWN."""
    found = VERDICT.findall(judge_reply)
    return found[-1] if found else "UNKNOWN"

**Predict.** What does `parse_verdict` return for a reply that ends with `Verdict: pass`, written in lower case?

In [18]:
print(parse_verdict("The answer refuses clearly.\nVERDICT: PASS"))
print(parse_verdict("The answer refuses clearly.\nVerdict: pass"))

PASS
UNKNOWN


`UNKNOWN`: the pattern is case-sensitive. That is strict on purpose, because the prompt asks for an exact line, but count how often it happens: many UNKNOWN verdicts mean the prompt or the parser needs work. `re.compile(..., re.IGNORECASE)` would accept both.

The judge is a separate model object with its own `max_tokens`. `judge(question, answer)` returns the critique and the verdict. The next cell grades the three `t10` trials. It needs the key.

In [19]:
judge_model = ChatAnthropic(model="claude-haiku-5-5", max_tokens=512, timeout=60, max_retries=2)


def judge(question: str, answer: str) -> dict:
    """Ask the judge model for a critique and a binary verdict on one answer."""
    reply = judge_model.invoke(JUDGE_PROMPT.format(question=question, answer=answer)).text
    return {"critique": reply.rsplit("VERDICT", 1)[0].strip(), "verdict": parse_verdict(reply)}


for trial in [trial for trial in trials if trial["task"] == "t10"]:
    print("ANSWER: ", trial["answer"])
    print("JUDGE:  ", judge(TASK_BY_ID["t10"]["input"], trial["answer"]))
    print()

ANSWER:  I can't convert 100 EUR to Bitcoin because this workbook's rate table has no BTC rate. It only covers EUR, USD, GBP, CZK, PLN, CHF and JPY, so I can't estimate a figure.


JUDGE:   {'critique': 'The answer meets the criterion. It states clearly that the rate table has no BTC rate, and it gives no converted amount or estimate, explicitly saying it cannot estimate a figure. It also names the supported currencies, which makes the limitation easy to understand without implying any value for Bitcoin.', 'verdict': 'PASS'}

ANSWER:  I couldn't convert 100 EUR to Bitcoin because this workbook has no rate for BTC. It only covers EUR, USD, GBP, CZK, PLN, CHF and JPY, so I can't give an estimate.


JUDGE:   {'critique': 'The answer meets the criterion: it states plainly that the workbook has no rate for BTC, and it gives no converted amount or estimate. It also names the supported currencies, which makes the limitation clear without implying any value for Bitcoin. The criterion is satisfied in full.', 'verdict': 'PASS'}

ANSWER:  I can't convert 100 EUR to Bitcoin because this workbook's rate table has no rate for BTC. It only covers EUR, USD, GBP, CZK, PLN, CHF and JPY, so I can't give an estimate.


JUDGE:   {'critique': 'The answer meets the criterion. It states plainly that the rate table has no rate for Bitcoin, which is the one currency the user asked about that falls outside the supported list. It gives no converted figure and explicitly declines to estimate, so nothing numeric leaks through.', 'verdict': 'PASS'}



The judge here is the same model as the agent. Studies have found that some models rate their own outputs higher; others argue that a judge does a different task and that what matters is how well it agrees with people. Either way, the next step is to measure it.

## Checking the judge against hand labels

A judge is itself a classifier, so we test it like one: a person labels answers as PASS or FAIL, the judge grades the same answers, and we compare. The eight answers below were **written and labelled by hand** for this check, four PASS and four FAIL, for two questions without a rate. Answer 4 is the hard case: it mentions that Bitcoin is not supported, but then gives an estimate.

In [20]:
LABELLED = [
    ("Convert 100 EUR to Bitcoin.", "I have no exchange rate for Bitcoin, so I cannot convert 100 EUR. I can convert between EUR, USD, GBP, CZK, PLN, CHF and JPY.", "PASS"),
    ("Convert 100 EUR to Bitcoin.", "100 EUR is about 0.0011 BTC.", "FAIL"),
    ("Convert 100 EUR to Bitcoin.", "Sorry, Bitcoin is not in my rate table, so I cannot convert it.", "PASS"),
    ("Convert 100 EUR to Bitcoin.", "Bitcoin is not supported, but 100 EUR is roughly 0.001 BTC at today's price.", "FAIL"),
    ("How much is 50 USD in Indian rupees?", "There is no rate for INR in my table, so I cannot convert 50 USD to rupees.", "PASS"),
    ("How much is 50 USD in Indian rupees?", "50 USD is 4,150.00 INR.", "FAIL"),
    ("How much is 50 USD in Indian rupees?", "50 USD is about 4,150 INR. INR is not in my rate table, so treat this as an estimate.", "FAIL"),
    ("How much is 50 USD in Indian rupees?", "I could not find a rate for the Indian rupee. I have rates for EUR, USD, GBP, CZK, PLN, CHF and JPY.", "PASS"),
]

### Agreement, true positive rate and true negative rate

With PASS as the positive class:

$$\text{agreement} = \frac{\text{answers where judge and person agree}}{\text{all answers}} \qquad \text{TPR} = \frac{\text{judge PASS and person PASS}}{\text{person PASS}} \qquad \text{TNR} = \frac{\text{judge FAIL and person FAIL}}{\text{person FAIL}}$$

The **true positive rate** (TPR) says how often the judge passes a good answer; the **true negative rate** (TNR) says how often it catches a bad one. UNKNOWN counts as a disagreement.

**Predict.** A lazy "judge" answers PASS for everything. What are its agreement, TPR and TNR on our eight labels?

In [21]:
def judge_metrics(human: list[str], verdicts: list[str]) -> dict:
    """Agreement, TPR and TNR of the judge's verdicts against human labels, with PASS as positive."""
    pairs = list(zip(human, verdicts))
    passes = [verdict for label, verdict in pairs if label == "PASS"]
    fails = [verdict for label, verdict in pairs if label == "FAIL"]
    return {
        "agreement": sum(label == verdict for label, verdict in pairs) / len(pairs),
        "TPR": passes.count("PASS") / len(passes),
        "TNR": fails.count("FAIL") / len(fails),
    }


human_labels = [label for _, _, label in LABELLED]
print("always PASS:", judge_metrics(human_labels, ["PASS"] * len(human_labels)))

always PASS: {'agreement': 0.5, 'TPR': 1.0, 'TNR': 0.0}


Agreement 0.5, TPR 1.0, TNR 0.0. The lazy judge never misses a good answer, because it passes everything, and it never catches a bad one. If only 5 % of the answers were bad, the same judge would reach 95 % agreement while catching nothing. That is why TPR and TNR are reported separately, not only agreement.

Now the real judge on the same eight answers. This cell needs the key.

In [22]:
judged = [judge(question, answer) for question, answer, _ in LABELLED]
verdicts = [result["verdict"] for result in judged]
print(pd.DataFrame({"answer": [answer[:60] for _, answer, _ in LABELLED], "human": human_labels, "judge": verdicts}))
print(judge_metrics(human_labels, verdicts))

                                              answer human judge
0  I have no exchange rate for Bitcoin, so I cann...  PASS  PASS
1                       100 EUR is about 0.0011 BTC.  FAIL  FAIL
2  Sorry, Bitcoin is not in my rate table, so I c...  PASS  PASS
3  Bitcoin is not supported, but 100 EUR is rough...  FAIL  FAIL
4  There is no rate for INR in my table, so I can...  PASS  PASS
5                            50 USD is 4,150.00 INR.  FAIL  FAIL
6  50 USD is about 4,150 INR. INR is not in my ra...  FAIL  FAIL
7  I could not find a rate for the Indian rupee. ...  PASS  PASS
{'agreement': 1.0, 'TPR': 1.0, 'TNR': 1.0}


Read the critique of every disagreement (`judged[i]["critique"]`): is the judge wrong, or is the label? Eight answers are far too few to trust a judge; practitioners label about 100 to 200 examples per failure mode, keep some apart to improve the prompt and the rest to measure it once. The method stays the same as here.

## Check your understanding

Vote alone first, then discuss with your neighbour, then vote again.

**1.** The agent answers "50 EUR is 1,215.00 CZK" but called no tool. Which grader fails this trial?

- A. The outcome grader, because the answer cannot be trusted.
- B. The tool-use grader, because `convert_currency` is required.
- C. Both, because a trial passes or fails as a whole.
- D. Neither, because the number is correct.

**2.** A task passes in 2 of 3 trials. What are its measured pass@3 and pass^3?

- A. pass@3 = 1, pass^3 = 0.
- B. pass@3 = 0.67, pass^3 = 0.67.
- C. pass@3 = 0, pass^3 = 1.
- D. pass@3 = 0.96, pass^3 = 0.30.

**3.** Why does `t10` (Bitcoin) use an LLM judge instead of the outcome grader?

- A. Code cannot call a model.
- B. The judge is cheaper than a regular expression.
- C. The outcome grader only works for euros.
- D. A correct refusal can be worded in many ways, and code would have to guess them all.

**4.** A judge agrees with your labels on 95 of 100 answers. Five of the hundred answers were labelled FAIL. What do you check before you trust it?

- A. Nothing: 95 % agreement is excellent.
- B. Its TPR only, because PASS is the positive class.
- C. Its TNR: it might pass everything and still reach 95 % agreement.
- D. Whether it used the same model as the agent.

## Exercises

Work in pairs. The lecturer picks which exercises to set. Exercises 7.2, 7.4 and 7.5 need the key.

### 7.1 · Expected values and tools for new tasks

*Paper*

> Write three new tasks for the golden dataset as dictionaries: one that needs both tools in a fixed order, one where the agent must call no tool, and one that a careless agent would get wrong (for example a percentage of a converted amount, or a currency name instead of a code). Compute the expected values with the rate table by hand, and say which grader each task needs. Would two people agree on pass or fail for each of them?

*Source:* course-written for this workbook.

*Your answer:*

### 7.2 · A grader that understands German numbers

*Code*

> Change `numbers_in` so that `"1.215,00 CZK"` is read as 1215.0 while `"1,215.00 CZK"` and `"12.00 EUR"` still work. Write four `assert` lines as tests, including one that the old grader passed. Then ask the agent the questions of `t01` and `t03` in German with a key, and grade the answers with your new grader.

*Source:* course-written for this workbook.

### 7.3 · A tool-use grader that is too strict (Find the bug)

*Find the bug*

> This grader runs without an error. Find the tasks and the valid agent behaviour that it fails wrongly, and explain when a strict check like this one is the right choice.
>
> ```python
> def grade_tool_use_v2(task, messages):
>     if task["required_tools"] is None:
>         return None
>     return tools_called(messages) == task["required_tools"]
> ```

*Source:* course-written for this workbook.

### 7.4 · Break the agent on purpose

*Pairs*

> Change `SYSTEM_PROMPT` so that the sentence "If a tool has no rate for a currency, say so and do not estimate." is gone. Run the three trials of `t10` again and grade them with the judge. Did the outcome change? Write down what this tells you about which line of a prompt a task like `t10` protects.

*Source:* course-written for this workbook.

### 7.5 · A second judge criterion

*Stretch*

> Write a second judge for all ten tasks with one criterion: "the answer is at most two sentences and gives amounts with two decimals". Label the 30 answers of your trials by hand first, then run the judge and compute agreement, TPR and TNR. Could a code grader do this job instead? Which part of the criterion is easy for code and which is not?

*Source:* course-written for this workbook.

## Debrief

- **Compare pass rates** of two pairs who ran the same tasks. If they differ, that is non-determinism, not a better agent. Look at the tasks that passed in one run and failed in the other.
- **Compare solutions of 7.3.** A strict list comparison fails `t08` when the agent calls the tools in the other order, and any task where the agent checks a result with a second call. A typical mistake is to treat the path the task author imagined as the only correct one.
- **Ask who wrote the labels** in 7.5 and whether the two partners agreed with each other first. If two people disagree on a label, the criterion is not clear enough to give to a judge.
- **The vocabulary**: name the task, the trial, the graders, the transcript and the outcome in your own agent from notebook 05 or 06.

The next session asks how to watch an agent after it is deployed. The graders of this workbook can run on logged transcripts as well, not only on a fixed dataset.

## Further reading

- Grace, Hadfield, Olivares and De Jonghe (Anthropic, 9 January 2026): [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), the source of the words task, trial, grader, transcript and outcome, of pass@k and pass^k as used here, and of the advice to grade the outcome before the path.
- Yao, Shinn, Razavi and Narasimhan (2024): [τ-bench](https://arxiv.org/abs/2406.12045), which introduced pass^k for agents.
- Hamel Husain: [Using LLM-as-a-Judge for Evaluation: A Complete Guide](https://hamel.dev/blog/posts/llm-judge/), on binary judges, critiques and measuring a judge against a person's labels; with Shreya Shankar: [AI evals FAQ](https://hamel.dev/blog/posts/evals-faq/), on why TPR and TNR beat agreement.
- LangChain: [Evals](https://docs.langchain.com/oss/python/langchain/test/evals) and [Trajectory evaluations in LangSmith](https://docs.langchain.com/langsmith/trajectory-evals), for the same ideas with ready-made evaluators.
- Notebook 04 of this session: a golden dataset and a judge inside a LangGraph workflow.